# Autoresearch Experiment Analysis

Analysis of autonomous loss-search results from `results.tsv`.

The notebook assumes the current `loss_suitability`-only results schema.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("results.tsv", sep="\t")

required_columns = ["commit", "loss_suitability", "status", "description"]
missing_columns = [name for name in required_columns if name not in df.columns]
if missing_columns:
    raise ValueError(
        f"results.tsv is missing required columns {missing_columns}; got {list(df.columns)}"
    )

metric_col = "loss_suitability"
metric_label = "Loss Suitability"
metric_short = "loss"

df[metric_col] = pd.to_numeric(df[metric_col], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"Primary metric: {metric_col} ({metric_label}, lower is better)")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    score = row[metric_col]
    desc = row["description"]
    print(f"  #{i:3d}  {metric_col}={score:.6f}  {desc}")

## Metric Over Time

Track how the best kept score evolves over time. The running minimum shows the current frontier, and lower is always better.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

valid = df[df["status"] != "CRASH"].copy()
valid = valid.reset_index(drop=True)

baseline_metric = valid.loc[0, metric_col]
window = max(1.0, abs(baseline_metric) * 0.25)
below = valid[valid[metric_col] <= baseline_metric + window]

disc = below[below["status"] == "DISCARD"]
ax.scatter(disc.index, disc[metric_col],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

kept_v = below[below["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v[metric_col],
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_metric = valid.loc[kept_mask, metric_col]
running_min = kept_metric.cummin()
ax.step(kept_idx, running_min, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

for idx, score in zip(kept_idx, kept_metric):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."

    ax.annotate(desc, (idx, score),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel(f"{metric_label} (lower is better)", fontsize=12)
ax.set_title(f"Autoresearch Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.2)

best_metric = kept_metric.min()
span = max(baseline_metric - best_metric, max(abs(baseline_metric) * 0.05, 1e-6))
margin = span * 0.15
ax.set_ylim(best_metric - margin, baseline_metric + margin)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
baseline_metric = df.iloc[0][metric_col]
best_metric = kept[metric_col].min()
best_row = kept.loc[kept[metric_col].idxmin()]

improvement = baseline_metric - best_metric
improvement_pct = (improvement / baseline_metric * 100.0) if abs(baseline_metric) > 1e-12 else np.nan

print(f"Baseline {metric_col}: {baseline_metric:.6f}")
print(f"Best {metric_col}:     {best_metric:.6f}")
print(f"Total improvement:    {improvement:.6f} ({improvement_pct:.2f}%)")
print(f"Best experiment:      {best_row['description']}")
print()

print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for _, row in kept_sorted.iterrows():
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: {metric_short}={row[metric_col]:.6f}  {desc}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
kept["prev_metric"] = kept[metric_col].shift(1)
kept["delta"] = kept["prev_metric"] - kept[metric_col]

hits = kept.iloc[1:].copy()
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>10}  {'Metric':>12}  Description")
print("-" * 90)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+10.6f}  {row[metric_col]:12.6f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+10.6f}  {'':>12}  TOTAL improvement over baseline")